In [0]:

from pyspark.sql.functions import current_timestamp, current_date
import re

# ----------------------------
# Config
# ----------------------------
CATALOG = "workspace"
TARGET_SCHEMA = "insurance_lakehouse"

VOLUME_PATH = "/Volumes/workspace/default/insurance_data"
POLICY_FILE = f"{VOLUME_PATH}/policy_data_databricks.csv"
CLAIMS_FILE = f"{VOLUME_PATH}/claims_data_databricks.csv"

BRONZE_POLICY_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.bronze_policy"
BRONZE_CLAIMS_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.bronze_claims"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {TARGET_SCHEMA}")

print("Using catalog/schema:", CATALOG, TARGET_SCHEMA)
print("Policy file:", POLICY_FILE)
print("Claims file:", CLAIMS_FILE)

In [0]:

# ----------------------------
# Helpers
# ----------------------------
def normalize_column_name(col_name: str) -> str:
    new_name = col_name.strip().lower()
    new_name = re.sub(r"[^a-zA-Z0-9]+", "_", new_name)
    new_name = re.sub(r"_+", "_", new_name)
    new_name = new_name.strip("_")
    return new_name if new_name else "col"

def normalize_columns(df):
    seen = {}
    for old_name in df.columns:
        new_name = normalize_column_name(old_name)

        if new_name in seen:
            seen[new_name] += 1
            new_name = f"{new_name}_{seen[new_name]}"
        else:
            seen[new_name] = 0

        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

    return df

def read_csv_to_bronze(path: str):
    # selectExpr("*", "_metadata.file_path as source_file") is often
    # the safest way to pull metadata columns from volume-backed reads.
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(path)
        .selectExpr("*", "_metadata.file_path as source_file")
        .withColumn("ingestion_ts", current_timestamp())
        .withColumn("snapshot_date", current_date())
    )

    df = normalize_columns(df)
    return df

In [0]:

# ----------------------------
# Load Policy Bronze
# ----------------------------
policy_bronze_df = read_csv_to_bronze(POLICY_FILE)

print("Policy Bronze schema:")
policy_bronze_df.printSchema()

display(policy_bronze_df.limit(10))

(
    policy_bronze_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_POLICY_TABLE)
)

print(f"Loaded {BRONZE_POLICY_TABLE}")

In [0]:

# ----------------------------
# Load Claims Bronze
# ----------------------------
claims_bronze_df = read_csv_to_bronze(CLAIMS_FILE)

print("Claims Bronze schema:")
claims_bronze_df.printSchema()

display(claims_bronze_df.limit(10))

(
    claims_bronze_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_CLAIMS_TABLE)
)

print(f"Loaded {BRONZE_CLAIMS_TABLE}")

In [0]:

# Validation
# ----------------------------
# display(spark.sql(f"SELECT COUNT(*) AS cnt FROM {BRONZE_POLICY_TABLE}"))
# display(spark.sql(f"SELECT COUNT(*) AS cnt FROM {BRONZE_CLAIMS_TABLE}"))

display(spark.sql(f"SELECT * FROM {BRONZE_POLICY_TABLE} LIMIT 5"))
display(spark.sql(f"SELECT * FROM {BRONZE_CLAIMS_TABLE} LIMIT 5"))